# Tutorial 07 - Patient-Level Summary for Downstream Analysis

This notebook aggregates FOOOF/specparam features into tidy segment-level and patient-level summaries, then exports files for future multi-patient analyses.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path('..') / 'outputs'
features_path = out_dir / 'fooof_features_segment_channel.csv'
selected_path = out_dir / 'selected_one_minute_segments.csv'

if not features_path.exists():
    raise FileNotFoundError(f'Missing feature file: {features_path}. Run Tutorial 06 first.')
if not selected_path.exists():
    raise FileNotFoundError(f'Missing segment metadata file: {selected_path}. Run Tutorial 05 first.')

feat_df = pd.read_csv(features_path)
sel_df = pd.read_csv(selected_path)

print('Feature rows:', len(feat_df))
print('Selected segments:', len(sel_df))
feat_df.head()

## 1) Build tidy segment-level summary

Each selected minute has channel-wise features. We aggregate across channels so each row represents one 1-minute segment.

In [ ]:
agg_map = {
    'aperiodic_offset': ['mean', 'median', 'std'],
    'aperiodic_exponent': ['mean', 'median', 'std'],
    'n_peaks': ['mean', 'median'],
    'top_peak_cf_hz': ['mean', 'median'],
    'top_peak_power': ['mean', 'median'],
    'fit_r2': ['mean', 'median', 'min'],
    'fit_error': ['mean', 'median', 'max']
}

seg_summary = feat_df.groupby('selection_rank').agg(agg_map)
seg_summary.columns = ['_'.join(col).strip() for col in seg_summary.columns.to_flat_index()]
seg_summary = seg_summary.reset_index()

seg_summary = seg_summary.merge(sel_df, on='selection_rank', how='left')
seg_summary.head()

## 2) Build patient-level summary

Here we collapse across all selected segments and channels to one patient-level row for this recording.

In [ ]:
patient_id = 'MGH4J_sid001_1d8_20130718_075948'

patient_summary = pd.DataFrame([{
    'patient_id': patient_id,
    'n_selected_segments': int(sel_df['selection_rank'].nunique()),
    'n_channel_fits': int(len(feat_df)),
    'aperiodic_exponent_mean': float(feat_df['aperiodic_exponent'].mean()),
    'aperiodic_exponent_std': float(feat_df['aperiodic_exponent'].std()),
    'aperiodic_offset_mean': float(feat_df['aperiodic_offset'].mean()),
    'fit_r2_median': float(feat_df['fit_r2'].median()),
    'fit_error_median': float(feat_df['fit_error'].median()),
    'n_peaks_mean': float(feat_df['n_peaks'].mean())
}])

patient_summary

## 3) Decide which features are suitable downstream

We make simple checks for missingness and unstable fit quality.

In [ ]:
quality_checks = pd.DataFrame({
    'feature': feat_df.columns,
    'missing_fraction': [feat_df[c].isna().mean() for c in feat_df.columns]
})

print('Missingness summary:')
display(quality_checks.sort_values('missing_fraction', ascending=False).head(10))

print('Rows with low fit_r2 (< 0.8):', int((feat_df['fit_r2'] < 0.8).sum()))
print('Rows with high fit_error (> 0.15):', int((feat_df['fit_error'] > 0.15).sum()))

In [ ]:
segment_level_path = out_dir / 'segment_level_feature_summary.csv'
patient_level_path = out_dir / 'patient_level_feature_summary.csv'
discussion_path = out_dir / 'patient_level_standardization_notes.md'

seg_summary.to_csv(segment_level_path, index=False)
patient_summary.to_csv(patient_level_path, index=False)

discussion = [
    '# Standardization Notes for Multi-Patient Analysis',
    '',
    '- Use the same sampling rate handling and resampling policy across patients.',
    '- Keep segment selection criteria fixed (stationarity thresholds, duration, spacing).',
    '- Use identical Welch and specparam/FOOOF parameter settings across all patients.',
    '- Track and filter poor fits consistently using common fit quality thresholds.',
    '- Harmonize channel montage naming and missing-channel handling before aggregation.',
    '- Preserve complete provenance (data version, code version, parameter files).'
]
discussion_path.write_text('\n'.join(discussion))

print('Saved:', segment_level_path.resolve())
print('Saved:', patient_level_path.resolve())
print('Saved:', discussion_path.resolve())

## Final recap

You now have:
- a channel-level feature table from model fits
- a segment-level tidy summary
- a patient-level summary row
- brief notes on what must be standardized for multi-patient work

This completes the single-patient tutorial sequence from data loading to export-ready features.